### Librerías

In [2]:
import os
import re

import pandas as pd
import numpy as np

import nltk
nltk.download('punkt_tab')
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mansu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
nltk.download('stopwords')
palabras_vacias = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mansu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Definición del gold standard

In [4]:
RUTA_CORPUS = './Corpus-Clustering/'
gold_standard : list[tuple[str, str]] = []

for etiqueta in os.listdir(RUTA_CORPUS):
	for documento in os.listdir(os.path.join(RUTA_CORPUS, etiqueta)):
		gold_standard.append((documento, etiqueta))

### Funciones de preprocesamiento

In [ ]:
# Definición de funciones básicas
def eliminar_cabecera(texto: list[str]) -> list[str]:
	"""
	Eliminación de la cabecera: todos los documentos tienen la cabecera
	en las n primeras líneas, aunque con n desconocido. Lo que sí se sabe
	es que todas las líneas de la cabecera van consecutivas y entre la 
	cabecera y el cuerpo del documento se pueden encontrar, al menos, una
	línea en blanco. Por ello, se buscará la aparición de la primera línea
	en blanco y se tomará el documento como que parte desde ahí.
	"""
	if len(texto) > 0:
		return texto[texto.index('\n')+1:]
	else:
		return []

def eliminar_lineas_blanco(texto: list[str]) -> list[str]:
	"""
	Eliminación de las líneas en blanco: todas las líneas que sean caracteres
	en blanco, como los espacios, los \n, tabuladores, etc. se eliminan, pues
	no aportan información.
	"""
	if len(texto) > 0:
		return [ linea for linea in texto if linea.strip() ]
	else:
		return []

def eliminar_firmas(texto: list[str]) -> list[str]:
	"""
	Eliminación de las firmas de los documentos: se puede observar que algunas
	de las firmas pueden aparecer explícitamente divididas del texto
	relevante del documento usando una fila de caracteres "-", "_", "=" o "+"
	seguidos de un posible espacio en blanco o los caracteres "\n" y/o "\r". 
	La firma debe ser el final del documento y, además, típicamente algo cortas. 
	Por ello se buscará la PRIMERA línea que cumpla la expresión regular: 
	^[ \t]*[-+=_]+[ \t\n\r]*$ y se comprobará si el número de líneas que le siguen a 
	partir de él no supera a las 10 líneas.

	Esta operación también puede provocar que se elimine texto que no se 
	considere una firma.
	"""
	if len(texto) == 0:
		return []

	for indice in range(0, len(texto)):
		if re.match('^[-+=_]+ *[\n\r]*$', texto[indice]):
			break
	
	# También se comprueba que no sea la última posición
	if len(texto[indice:]) <= 10 and indice != len(texto)-1:
		texto_intermedio = texto[:indice]
	else:
		texto_intermedio = texto
	
	return texto_intermedio
	
def eliminar_caracteres_blanco_puntuacion(texto: list[str]) -> list[str]:
	"""
	Eliminación de caracteres en blanco y signos de puntuación: los signos de 
	puntuación genéricos.
	"""
	if len(texto) > 0:
		return [re.sub(r'[\^+_\-\.:,;><~\?\!\(\)]+', '', linea.strip()) for linea in texto]
	else:
		return []
	
def eliminar_correos_electronicos(texto: list[str]) -> list[str]:
	if len(texto) > 0:
		return [re.sub(r'\w+(\.\w+)*@\w+\.\w+(\.\w+)*', '', linea.strip()) for linea in texto]
	else:
		return []

def eliminar_numeros(texto: list[str]) -> list[str]:
	if len(texto) > 0:
		return [re.sub(r'\d+', '', linea) for linea in texto]
	else:
		return []

def conversion_minusculas(texto: str) -> str:
	"""
	Paso de mayúsculas a minúsculas en el texto.
	"""
	return texto.lower()

### Funciones principales

In [ ]:
def contar_palabras(texto_original: list[str]) -> int:
	"""
	Elimina las caracteres en blanco y los signos de puntuación del texto y cuenta
	cuantas palabras hay
	"""
	return len(
		tokenizado(
			' '.join(eliminar_caracteres_blanco_puntuacion(
				eliminar_lineas_blanco(texto_original)
			))
		)
	)

def preprocesado(texto_original: list[str]) -> str:

	# print('Original', texto_original)
	
	texto_intermedio = eliminar_cabecera(texto_original)
	# print('Paso 1', texto_intermedio)

	texto_intermedio = eliminar_lineas_blanco(texto_intermedio)
	# print('Paso 2', texto_intermedio)

	texto_intermedio = eliminar_firmas(texto_intermedio)
	# print('Paso 3', texto_intermedio)

	texto_intermedio = eliminar_correos_electronicos(texto_intermedio)

	texto_intermedio = eliminar_numeros(texto_intermedio)
	
	texto_intermedio = eliminar_caracteres_blanco_puntuacion(texto_intermedio)
	# print('Paso 4', texto_intermedio)

	texto_intermedio = ' '.join(texto_intermedio)

	# 5. Conversión de mayúsculas a minúsculas
	texto_intermedio = conversion_minusculas(texto_intermedio)
	# print('Paso 5', texto_intermedio)

	return texto_intermedio

def eliminar_palabras_vacias(texto_original: list[str]) -> str:
	"""
	Elimina las palabras vacías que se encuentren en el texto según el conjunto
	definido en `palabras_vacias` 
	"""
	return ' '.join([ palabra for palabra in texto_original if palabra not in palabras_vacias ])

def tokenizado(texto: str) -> list[str]:
	"""
	División del texto por palabras
	"""
	return re.split(' +', texto)

def stemmer(texto_original: str):
	"""
	Aplicación del stemmer de Porter sobre el texto
	"""
	ps = PorterStemmer()
	texto_stemmer = word_tokenize(texto_original)

	texto_final = []

	for palabra in texto_stemmer:
		texto_final.append(ps.stem(palabra))

	return ' '.join(texto_final)

def calcular_frecuencia(texto: str):
	frecuencia_rasgos: dict[str, int] = {}

	for rasgo in texto.split(' '):
		if rasgo not in frecuencia_rasgos:
			frecuencia_rasgos[rasgo] = 0
		frecuencia_rasgos[rasgo] += 1
	
	return frecuencia_rasgos

### Conteo de palabras

In [7]:
# Número de palabras de cada documento por categoría.
conteo_palabras: dict[str, list[int]] = {}

for par in gold_standard:

	if par[1] not in conteo_palabras:
		conteo_palabras[par[1]] = []

	with open(os.path.join(RUTA_CORPUS, par[1], par[0]), 'r') as fp:
		texto = fp.readlines()

	conteo_palabras[par[1]].append(contar_palabras(texto))

# Una vez terminado hay que recorrer cada clase para calcular la media y la 
# desviación
for clase, lista in conteo_palabras.items():
	print(f"Clase {clase}:")
	print(f"Mínimo: {np.min(lista)}")
	print(f"Máximo: {np.max(lista)}")
	print(f"Media: {round(np.mean(lista), 2)}")
	print(f"Desviación típica: {round(np.std(lista), 2)}")

Clase comp.sys.ibm.pc.hardware:
Mínimo: 47
Máximo: 4056
Media: 240.77
Desviación típica: 363.74
Clase comp.sys.mac.hardware:
Mínimo: 28
Máximo: 689
Media: 180.89
Desviación típica: 106.22
Clase rec.autos:
Mínimo: 30
Máximo: 828
Media: 211.59
Desviación típica: 165.4
Clase rec.sport.hockey:
Mínimo: 32
Máximo: 1479
Media: 266.46
Desviación típica: 255.7
Clase sci.electronics:
Mínimo: 47
Máximo: 11716
Media: 258.51
Desviación típica: 798.9
Clase talk.politics.guns:
Mínimo: 66
Máximo: 3866
Media: 404.95
Desviación típica: 468.54
Clase talk.politics.mideast:
Mínimo: 54
Máximo: 10160
Media: 713.54
Desviación típica: 1293.97


### Clustering

In [ ]:
frecuencia_documento_rasgo: dict[str, dict[str, int]] = {}

for par in gold_standard:
	documento = par[0]
	etiqueta = par[1]

	with open(os.path.join(RUTA_CORPUS, etiqueta, documento), 'r') as fp:
		texto = fp.readlines() 

	texto_preprocesado = preprocesado(texto)

	if texto_preprocesado:

		texto_tokenizado = tokenizado(texto_preprocesado)

		texto_limpio = eliminar_palabras_vacias(texto_tokenizado) 

		texto_stemmer = stemmer(texto_limpio)
		
		frecuencia_documento_rasgo[documento] = calcular_frecuencia(texto_stemmer)

In [ ]:
frecuencia_rasgo_documento: dict[str, set] = {}

# Cálculo del DF
for documento, rasgos in frecuencia_documento_rasgo.items():
	for rasgo in rasgos.keys():
		if rasgo not in frecuencia_rasgo_documento:
			frecuencia_rasgo_documento[rasgo] = set()
		frecuencia_rasgo_documento[rasgo].add(documento)

# Eliminación de los rasgos que aparezcan en muy pocos documentos (mínimo 6)
frecuencia_rasgo_documento_reducido = {}
for rasgo, conjunto in frecuencia_rasgo_documento.items():
	if len(conjunto) >= 7:
		frecuencia_rasgo_documento_reducido[rasgo] = conjunto

del frecuencia_rasgo_documento

Creación de las matrices

In [ ]:
matriz_frecuencias = pd.DataFrame(data=[], index=(frecuencia_rasgo_documento_reducido.keys()))

for documento, rasgos in frecuencia_documento_rasgo.items():
    matriz_frecuencias[documento] = pd.Series(rasgos)

matriz_frecuencias = matriz_frecuencias.fillna(0)

# Rasgos reducidos por documento
frecuencia_documento_rasgo = matriz_frecuencias.to_dict()

# matriz_tf_idf = lo que sea 

C:\Users\mansu\AppData\Local\Temp\ipykernel_6132\4069200626.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matriz_frecuencias[documento] = pd.Series(rasgos)
C:\Users\mansu\AppData\Local\Temp\ipykernel_6132\4069200626.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matriz_frecuencias[documento] = pd.Series(rasgos)
C:\Users\mansu\AppData\Local\Temp\ipykernel_6132\4069200626.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance